## Modelos De Regresión y Clasificación


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

#Modelos de ML
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
#StandarScaler lo que hace es escalar y transformar las features a media = 0 y desviacion = 1
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

#Metricas de evaluación
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
df = pd.read_csv('student_habits_performance.csv')

df = df.drop(columns=['student_id'])

df.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            1000 non-null   int64  
 1   gender                         1000 non-null   str    
 2   study_hours_per_day            1000 non-null   float64
 3   social_media_hours             1000 non-null   float64
 4   netflix_hours                  1000 non-null   float64
 5   part_time_job                  1000 non-null   str    
 6   attendance_percentage          1000 non-null   float64
 7   sleep_hours                    1000 non-null   float64
 8   diet_quality                   1000 non-null   str    
 9   exercise_frequency             1000 non-null   int64  
 10  parental_education_level       909 non-null    str    
 11  internet_quality               1000 non-null   str    
 12  mental_health_rating           1000 non-null   int64  
 13  

In [5]:
#Tratamiento de nulos

moda_educacion = df['parental_education_level'].mode()[0]

df['parental_education_level'] = df['parental_education_level'].fillna(moda_educacion)

In [6]:
df.isnull().sum()

age                              0
gender                           0
study_hours_per_day              0
social_media_hours               0
netflix_hours                    0
part_time_job                    0
attendance_percentage            0
sleep_hours                      0
diet_quality                     0
exercise_frequency               0
parental_education_level         0
internet_quality                 0
mental_health_rating             0
extracurricular_participation    0
exam_score                       0
dtype: int64

In [7]:
# OrdinalEncoder para variables que tienen orden real

#Dieta
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)


#Internet
oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)


#Educación
oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)


#Binario de trabajo de medio tiempo
df['part_time_job'] = (df['part_time_job'] == 'Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] == 'Yes').astype(int)


#One Hot Encoding para gender

df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int )

df.dtypes


age                                int64
study_hours_per_day              float64
social_media_hours               float64
netflix_hours                    float64
part_time_job                      int64
attendance_percentage            float64
sleep_hours                      float64
diet_quality                     float64
exercise_frequency                 int64
parental_education_level         float64
internet_quality                 float64
mental_health_rating               int64
extracurricular_participation      int64
exam_score                       float64
gender_Male                        int64
gender_Other                       int64
dtype: object

In [8]:
#División del dataset en entrenamineto y prueba
#Definir un alista de todas las features que el modelo usará para la regresión
#Excluimos a exam_score ya que ella es el target que queemos predecir

features_reg = ['study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 
                'attendance_percentage', 'sleep_hours','diet_quality', 'exercise_frequency', 
                'parental_education_level', 'internet_quality', 'mental_health_rating', 
                'extracurricular_participation','gender_Male', 'gender_Other']

X_reg = df[features_reg]

y_reg = df['exam_score']

X_train_r, X_test_r, y_train_r,y_test_r = train_test_split(
    X_reg, y_reg, 
    test_size=0.2, 
    random_state=42
)

scaler_r = StandardScaler()

X_train_r_sc = scaler_r.fit_transform(X_train_r)

X_test_r_sc = scaler_r.transform(X_test_r)

#Verficar el tamaño de cada conjunto resultante
print(f'Entreamiento: {X_train_r_sc.shape[0]} estudiantes | prueba: {X_test_r_sc.shape[0]} estudiantes')


Entreamiento: 800 estudiantes | prueba: 200 estudiantes


In [9]:
modelo_lr = LinearRegression()

modelo_lr.fit(X_train_r_sc, y_train_r)

y_pred_lr = modelo_lr.predict(X_test_r_sc)

pd.DataFrame({
    'Puntaje real': y_test_r.values[:8],
    'Puntaje predicho': y_pred_lr[:8].round(1)
})

,Puntaje real,Puntaje predicho
0,64.2,65.9
1,72.7,74.7
2,79.0,78.5
3,79.5,73.5
4,58.2,61.2
5,53.4,54.8
6,70.8,75.4
7,62.5,55.3


In [10]:
mae = mean_absolute_error(y_test_r, y_pred_lr)

mse = mean_squared_error(y_test_r, y_pred_lr)

rmse = np.sqrt(mse)

r2 = r2_score(y_test_r, y_pred_lr)

pd.DataFrame({
    'Metrica': ['MAE', 'RMSE', 'R´2'],
    'Valor': [round(mae, 3), round(rmse, 3), round(r2, 3)],
    'Interepretación': [
        f'Error promedio: +-{mae:.1f} puntos por estudiantes',
        f'Error (penalizando outliers): +-{rmse:.1f} puntos',
        f'Explica el {r2*100:.1f}% de la variabilidad en los puntajes'
    ]
})

,Metrica,Valor,Interepretación
0,MAE,4.124,Error promedio: +-4.1 puntos por estudiantes
1,RMSE,5.076,Error (penalizando outliers): +-5.1 puntos
2,R´2,0.900,Explica el 90.0% de la variabilidad en los pun...
